# Preprocess - Hospital Readmission

Cleans `Dataset/hospital-readmission/diabetic_data.csv` and saves `Dataset/processed/hospital_readmission_clean.csv` for the train notebook.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Dataset/hospital-readmission/diabetic_data.csv")
df = df.replace("?", np.nan)
print("Raw shape:", df.shape)

In [ ]:
# one encounter per patient, otherwise repeat admissions of the same person leak into train and test
df = df.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
print("After dedup to one encounter per patient:", df.shape)

In [ ]:
# id columns aren't predictive, weight/payer_code are almost entirely missing
df = df.drop(columns=["encounter_id", "patient_nbr", "weight", "payer_code"])

near_constant = [c for c in df.columns if df[c].nunique() <= 1]
df = df.drop(columns=near_constant)
print("Dropped near-constant columns:", near_constant)

In [ ]:
# medical_specialty has heavy missingness, keep the common ones and bucket the rest
top_specialties = df["medical_specialty"].value_counts().nlargest(10).index
df["medical_specialty"] = df["medical_specialty"].where(
    df["medical_specialty"].isin(top_specialties), "Other/Unknown"
).fillna("Other/Unknown")

df["race"] = df["race"].fillna("Unknown")
df = df[df["gender"] != "Unknown/Invalid"]

In [ ]:
def icd9_group(code):
    # simplified diagnosis grouping following the same logic used in Strack et al. (2014)
    if pd.isna(code):
        return "Missing"
    if str(code).startswith("V") or str(code).startswith("E"):
        return "Other"
    try:
        c = float(code)
    except ValueError:
        return "Other"
    if 390 <= c <= 459 or c == 785:
        return "Circulatory"
    if 460 <= c <= 519 or c == 786:
        return "Respiratory"
    if 520 <= c <= 579 or c == 787:
        return "Digestive"
    if 250 <= c < 251:
        return "Diabetes"
    if 800 <= c <= 999:
        return "Injury"
    if 710 <= c <= 739:
        return "Musculoskeletal"
    if 580 <= c <= 629 or c == 788:
        return "Genitourinary"
    if 140 <= c <= 239:
        return "Neoplasms"
    return "Other"


for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].apply(icd9_group)

In [ ]:
# target: early readmission (<30 days) vs not
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])

print("Target balance:")
print(df["readmitted_30"].value_counts(normalize=True))

In [ ]:
df.to_csv("../Dataset/processed/hospital_readmission_clean.csv", index=False)
print("Saved", df.shape, "to Dataset/processed/hospital_readmission_clean.csv")